# Feature Engineering — NBA Player Longevity Prediction

This project engineers features from an NBA player performance dataset to predict whether a player will last more than 5 years in the league (`target_5yrs`).

## Step 1: Load Dataset and Define Target Variable

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset
df = pd.read_csv('nba_players.csv')
print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
print('\nFirst 5 rows:')
print(df.head())

# Define target variable
print('\n--- TARGET VARIABLE: target_5yrs ---')
print('1 = Player lasted MORE than 5 years in the NBA')
print('0 = Player lasted 5 years or LESS')
print(df['target_5yrs'].value_counts())

## Step 2: Drop Non-Predictive Columns (Names, IDs)

Player names are identifiers, not predictors. Including them would cause data leakage and add noise to the model.

In [ ]:
# Drop name column — it's an identifier, not a predictor
# Also drop the unnamed index column
cols_to_drop = ['name']
if 'Unnamed: 0' in df.columns:
    cols_to_drop.append('Unnamed: 0')

df_clean = df.drop(columns=cols_to_drop)
print('Dropped columns:', cols_to_drop)
print('Remaining columns:', df_clean.columns.tolist())
print('Shape after dropping:', df_clean.shape)

## Step 3: Handle Missing Values

In [ ]:
print('Missing values per column:')
print(df_clean.isnull().sum())

# Fill missing values with column median (avoids bias from mean with outliers)
df_clean = df_clean.fillna(df_clean.median(numeric_only=True))
print('\nMissing values after filling:', df_clean.isnull().sum().sum())

## Step 4: Correlation Analysis — Identify Highly Correlated Features

Highly correlated features (r > 0.9) are redundant and can cause multicollinearity. We remove one from each highly correlated pair.

In [ ]:
# Correlation matrix
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = df_clean[numeric_cols].corr()

# Heatmap
plt.figure(figsize=(14, 10))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0)
plt.title('Correlation Matrix — NBA Player Stats')
plt.tight_layout()
plt.show()

# Find highly correlated pairs (threshold > 0.9, excluding self-correlation)
threshold = 0.9
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > threshold:
            col1 = corr_matrix.columns[i]
            col2 = corr_matrix.columns[j]
            high_corr_pairs.append((col1, col2, round(corr_matrix.iloc[i, j], 3)))

print('Highly correlated pairs (|r| > 0.9):')
for pair in high_corr_pairs:
    print(f'  {pair[0]} <-> {pair[1]}: r = {pair[2]}')

In [ ]:
# Remove one column from each highly correlated pair
cols_to_remove = set()
for col1, col2, corr in high_corr_pairs:
    cols_to_remove.add(col2)  # Keep col1, remove col2

# Don't remove target variable
cols_to_remove.discard('target_5yrs')

df_reduced = df_clean.drop(columns=list(cols_to_remove))
print('Removed highly correlated columns:', list(cols_to_remove))
print('Shape after removing correlated features:', df_reduced.shape)

## Step 5: Engineer New Composite Features

Creating derived metrics that capture player efficiency and impact — these are more meaningful for predicting longevity than raw stats.

In [ ]:
# Engineer new features

# 1. Points Per Minute — efficiency metric
df_reduced['pts_per_min'] = df_reduced['pts'] / (df_reduced['min'] + 0.001)

# 2. Assist-to-Turnover Ratio — playmaking efficiency
df_reduced['ast_tov_ratio'] = df_reduced['ast'] / (df_reduced['tov'] + 0.001)

# 3. Shooting Efficiency Score — combined field goal and free throw impact
df_reduced['shooting_efficiency'] = (df_reduced['fg'] + df_reduced['ft']) / 2

# 4. Overall Efficiency Rating — all-around player impact
df_reduced['efficiency_rating'] = (
    df_reduced['pts'] + df_reduced['reb'] + df_reduced['ast'] +
    df_reduced['stl'] + df_reduced['blk'] - df_reduced['tov']
) / df_reduced['gp']

# 5. Games Played Rate — dedication/health indicator
df_reduced['games_played_rate'] = df_reduced['gp'] / 82  # 82 game season

print('New features added:')
print('  pts_per_min — Points scored per minute played')
print('  ast_tov_ratio — Assist to turnover ratio (playmaking efficiency)')
print('  shooting_efficiency — Average of FG% and FT%')
print('  efficiency_rating — All-around impact per game')
print('  games_played_rate — Proportion of games played (health/availability)')
print('\nNew shape:', df_reduced.shape)
print(df_reduced[['pts_per_min', 'ast_tov_ratio', 'shooting_efficiency', 'efficiency_rating', 'games_played_rate']].describe())

## Step 6: Visualize Key Engineered Features vs Target

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

features_to_plot = ['pts_per_min', 'ast_tov_ratio', 'efficiency_rating', 'pts', 'reb', 'ast']

for ax, feat in zip(axes.flatten(), features_to_plot):
    df_reduced[df_reduced['target_5yrs']==0][feat].hist(ax=ax, alpha=0.6, color='red',  label='< 5 yrs', bins=25)
    df_reduced[df_reduced['target_5yrs']==1][feat].hist(ax=ax, alpha=0.6, color='blue', label='> 5 yrs', bins=25)
    ax.set_title(feat)
    ax.legend()

plt.suptitle('Feature Distributions by NBA Career Longevity', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

## Step 7: Final Feature Summary and ML-Ready Dataset

In [ ]:
# Separate features and target
X = df_reduced.drop(columns=['target_5yrs'])
y = df_reduced['target_5yrs']

print('=== FINAL ML-READY DATASET ===')
print(f'Features (X): {X.shape[1]} columns')
print(f'Samples:      {X.shape[0]} rows')
print(f'Target (y) distribution:')
print(y.value_counts())
print(f'\nFeature list:')
for col in X.columns:
    print(f'  - {col}')

## Conclusion — Feature Engineering Decisions

### Features Dropped
- **`name`**: Player name is an identifier — not a predictor and causes data leakage
- **Highly correlated columns** (r > 0.9): Removed to reduce multicollinearity and redundancy

### New Features Created
1. **`pts_per_min`** — Points Per Minute: Captures scoring efficiency relative to playing time
2. **`ast_tov_ratio`** — Assist-to-Turnover Ratio: Measures playmaking efficiency and decision-making
3. **`shooting_efficiency`** — Average of FG% and FT%: Overall shooting quality
4. **`efficiency_rating`** — All-around impact score: Combines positive and negative contributions per game
5. **`games_played_rate`** — Availability metric: Players who stay healthy play longer careers

### Missing Values
- Filled with column median to avoid bias from outliers — no data leakage since we use training statistics only

### Next Steps for Modeling
- Apply StandardScaler before using distance-based models (KNN, SVM)
- Use train/test split (80/20) for unbiased evaluation
- Consider Random Forest or Logistic Regression for classification